# Notebook Header & Imports

In [ ]:
# EEEM068: Industrial Waste Classification
# Training Notebook (Jupyter version of train.py)

# Standard library imports — used for file management, configuration handling,
# reproducibility utilities, and timing within the training loop.
import os
import json
import copy
import argparse
import random
import time
from pathlib import Path

# External libraries — core scientific stack for deep learning, metrics, and visualisation.
import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets, models
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar — improves training transparency and helps monitor convergence behaviour.
from tqdm.auto import tqdm

# Project utilities — modularised components ensure reproducibility and keep the notebook clean.
from utils.data import get_transforms
from utils.models import get_model
from utils.plotting import save_confusion_matrix
# These external versions replace inline definitions to maintain a single source of truth.
#from utils.config import load_config, save_config, deep_merge
#from utils.data import get_dataloaders
#from utils.models import get_model
#from utils.optim import get_optimizer, get_scheduler
#from utils.plotting import save_loss_curves, save_confusion_matrix, save_metrics
#from utils.repro import set_seed

# Config Functions

In [ ]:
# Merge two config dictionaries, giving priority to values in `override`
def deep_merge(base: dict, override: dict) -> dict:
    # Deep‑copy avoids mutating the original base config, preserving reproducibility
    result = copy.deepcopy(base)
    for key, value in override.items():
        if key == "base":  # prevents recursive inheritance loops
            continue
        # Nested dictionaries are merged recursively so hierarchical configs behave predictably
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        else:
            # Override values take precedence, enabling experiment‑specific customisation
            result[key] = value
    return result


# Load an experiment config, apply inheritance, and apply any CLI overrides
def load_config(experiment_path: str, cli_overrides: dict = None) -> dict:
    with open(experiment_path) as f:
        experiment = yaml.safe_load(f)

    # Config inheritance allows shared defaults while keeping experiments lightweight and DRY
    if "base" in experiment:
        with open(experiment["base"]) as f:
            base = yaml.safe_load(f)
        config = deep_merge(base, experiment)
    else:
        config = experiment

    # CLI overrides enable quick adjustments (e.g., LR, epochs) without editing YAML files
    if cli_overrides:
        for key, value in cli_overrides.items():
            if value is not None:
                config["training"][key] = value

    return config


# Save the final merged config to the output directory
def save_config(config: dict, output_dir: str):
    # Saving the resolved config ensures full reproducibility of the training run
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "config.yaml"), "w") as f:
        yaml.dump(config, f)

# Seed Function

In [ ]:
# Set all random seeds to ensure reproducible training runs
def set_seed(seed: int):
    # Seeding all major RNGs ensures that data shuffling, weight initialisation,
    # and augmentation behave identically across repeated runs.
    random.seed(seed)                     # Python RNG
    np.random.seed(seed)                  # NumPy RNG
    torch.manual_seed(seed)               # PyTorch CPU RNG
    torch.cuda.manual_seed_all(seed)      # PyTorch GPU RNG

    # Enforcing deterministic CUDA behaviour removes nondeterministic kernel choices,
    # trading a small amount of speed for fully reproducible results.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"[Seed] Set to {seed}")

# Transforms + DataLoaders

In [ ]:
# Build training or evaluation transforms
'''
def get_transforms(config: dict, train: bool = True):
    aug = config["augmentation"]
    img_size = (config["dataset"]["img_height"], config["dataset"]["img_width"])

    # Normalisation aligns inputs with ImageNet‑pretrained backbones,
    # ensuring stable activations and preventing brightness/exposure bias.
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    if train:
        # Augmentation improves generalisation by exposing the model to realistic
        # variations in geometry and colour, reducing overfitting.
        tf_list = [
            transforms.RandomResizedCrop(img_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ColorJitter(
                brightness=aug["brightness"],
                contrast=aug["contrast"],
                saturation=aug["saturation"],
                hue=aug["hue"]
            ),
            transforms.ToTensor(),
            normalize
        ]

        # Random erasing simulates occlusion and encourages the model to rely on
        # distributed features rather than a single discriminative region.
        if aug.get("random_erasing", False):
            tf_list.append(
                transforms.RandomErasing(
                    p=0.25,
                    scale=(0.02, 0.2),
                    ratio=(0.3, 3.3),
                    value='random'
                )
            )

        return transforms.Compose(tf_list)

    # Validation/test transforms avoid augmentation to ensure unbiased, repeatable evaluation.
    return transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        normalize
    ])
'''

# Create train, validation, and test dataloaders
def get_dataloaders(config: dict):
    train_tf = get_transforms(config, train=True)
    val_tf   = get_transforms(config, train=False)

    data_root  = config["dataset"]["data_root"]
    batch_size = config["training"]["batch_size"]
    seed       = config["dataset"]["seed"]
    use_sampler = config["training"].get("use_weighted_sampler", False)

    # ImageFolder automatically assigns labels based on directory structure,
    # ensuring consistent class indexing across splits.
    train_dataset = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    val_dataset   = datasets.ImageFolder(os.path.join(data_root, "val"),   transform=val_tf)
    test_dataset  = datasets.ImageFolder(os.path.join(data_root, "test"),  transform=val_tf)

    # Weighted sampler corrects class imbalance by increasing sampling probability
    # for minority classes, preventing the model from collapsing to majority predictions.
    sampler = None
    if use_sampler:
        targets = train_dataset.targets
        class_counts = np.bincount(targets, minlength=len(train_dataset.classes))
        class_weights = 1.0 / (class_counts + 1e-6)
        sample_weights = class_weights[targets]

        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(sample_weights).float(),
            num_samples=len(sample_weights),
            replacement=True
        )

        print(f"[Imbalance Fix] WeightedRandomSampler ENABLED ({len(train_dataset.classes)} classes)")
    else:
        print("[Imbalance Fix] Sampler DISABLED (Phase 1)")

    # Windows requires workers=0; pin_memory accelerates host→GPU transfer when CUDA is available.
    pin = torch.cuda.is_available()
    workers = 0 if os.name == "nt" else 4

    # Training loader: shuffle only when not using a sampler, since sampler defines sampling order.
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=not use_sampler,
        sampler=sampler,
        num_workers=workers,
        pin_memory=pin
    )

    # Validation loader: deterministic ordering ensures stable evaluation.
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=pin
    )

    # Test loader may use a different batch size for deployment‑style evaluation.
    test_loader = DataLoader(
        test_dataset,
        batch_size=config["evaluation"]["batch_size"],
        shuffle=False,
        num_workers=workers,
        pin_memory=pin
    )

    print(f"[Data] Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}")
    return train_loader, val_loader, test_loader

# Model Builder

In [ ]:
# Build a classification model based on the selected backbone
'''
def get_model(config: dict) -> nn.Module:
    backbone   = config["run"]["model"]
    n_classes  = config["dataset"]["n_classes"]
    pretrained = config["training"]["pretrained"]

    # Using ImageNet weights accelerates convergence and improves performance
    # when the target dataset is relatively small or visually similar.
    weights = "IMAGENET1K_V1" if pretrained else None

    print(f"[Model] Loading {backbone} (pretrained={pretrained})")

    # Replacing only the classification head preserves the pretrained feature extractor.
    # This is the core principle of transfer learning: reuse general visual features
    # while adapting the final layer to the specific number of classes.
    if backbone == "resnet50":
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, n_classes)

    elif backbone == "efficientnet_b3":
        model = models.efficientnet_b3(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, n_classes)

    elif backbone == "swin_t":
        model = models.swin_t(weights=weights)
        model.head = nn.Linear(model.head.in_features, n_classes)

    elif backbone == "convnext_t":
        model = models.convnext_tiny(weights=weights)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, n_classes)

    else:
        raise ValueError(
            f"Unknown model: {backbone}. "
            "Choose from: resnet50, efficientnet_b3, swin_t, convnext_t"
        )

    # Parameter counts help verify the fine‑tuning strategy.
    # A large gap between total and trainable parameters indicates frozen layers,
    # which is useful when applying staged learning rates or partial fine‑tuning.
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Model] Params — total: {total_params:,} | trainable: {trainable_params:,}")

    return model
'''

# Optimizer + Scheduler

In [ ]:
# Create an optimizer, optionally using staged learning rates for fine‑tuning
def get_optimizer(model, config):
    opt_cfg = config["optimizer"]
    ft_cfg  = config["fine_tuning"]
    lr      = config["training"]["learning_rate"]

    # Staged LR applies a smaller LR to pretrained backbone layers and a larger LR
    # to newly‑initialised classification layers. This prevents catastrophic forgetting
    # while still allowing the new head to adapt quickly.
    if ft_cfg["staged_lr"]:
        new_layer_names = ft_cfg.get("new_layers", ["fc", "classifier", "head"])
        head_params, backbone_params = [], []

        # Separate parameters so each group can receive its own LR multiplier.
        # This is essential for controlled fine‑tuning of pretrained models.
        for name, param in model.named_parameters():
            if any(layer in name for layer in new_layer_names):
                head_params.append(param)
            else:
                backbone_params.append(param)

        param_groups = [
            {"params": backbone_params, "lr": lr * ft_cfg["base_lr_mult"]},
            {"params": head_params,     "lr": lr}
        ]
    else:
        # Uniform LR is appropriate when training from scratch or when full fine‑tuning is desired.
        param_groups = model.parameters()

    # Optimizer choice affects convergence behaviour:
    # Adam adapts per‑parameter learning rates and is robust for noisy gradients,
    # while SGD with momentum often yields better generalisation on vision tasks.
    if opt_cfg["type"] == "adam":
        return torch.optim.Adam(
            param_groups,
            lr=lr,
            betas=(opt_cfg["adam_beta1"], opt_cfg["adam_beta2"]),
            weight_decay=opt_cfg["weight_decay"]
        )
    else:
        return torch.optim.SGD(
            param_groups,
            lr=lr,
            momentum=opt_cfg["momentum"],
            dampening=opt_cfg["sgd_dampening"],
            nesterov=opt_cfg["sgd_nesterov"],
            weight_decay=opt_cfg["weight_decay"]
        )


# Create a learning‑rate scheduler based on the config
def get_scheduler(optimizer, config):
    sched_type = config["scheduler"]["type"]
    ft_cfg = config["fine_tuning"]

    # Cosine annealing provides a smooth, non‑monotonic decay that helps escape shallow minima
    # and is widely used in modern vision training pipelines.
    if sched_type == "CosineAnnealingLR":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=ft_cfg["T_max"]
        )

    # Multi‑step decay performs abrupt LR drops at predefined milestones,
    # which can stabilise training when the loss plateaus.
    return torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=ft_cfg["stepsize"],
        gamma=ft_cfg["gamma"]
    )

# Training + Validation

In [ ]:
# Train the model for one epoch
def train_one_epoch(model, loader, criterion, optimizer, device, log_interval,
                    scaler, use_amp):

    # Training mode enables dropout and batchnorm updates for correct optimisation behaviour
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    # A progress bar helps monitor convergence and detect instability early
    pbar = tqdm(enumerate(loader), total=len(loader), desc="  Training", leave=False)

    for batch_idx, (images, labels) in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)  # avoids gradient accumulation and improves memory efficiency

        # Mixed precision reduces memory use and speeds up training on modern GPUs
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Gradient scaling prevents underflow in FP16 gradients, stabilising AMP training
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Track running loss and accuracy for monitoring training dynamics
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

        # Periodic logging helps diagnose divergence or plateauing during long runs
        if (batch_idx + 1) % log_interval == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total, correct / total


# Evaluate the model without gradient updates
@torch.no_grad()
def validate(model, loader, criterion, device):
    # Evaluation mode disables dropout/batchnorm updates for deterministic behaviour
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        # Accumulate loss and accuracy to assess generalisation performance
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

        # Store predictions for F1 and confusion matrix analysis
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # Macro F1 treats all classes equally, making it robust to imbalance
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return total_loss / total, correct / total, f1, all_preds, all_labels

# Plotting Functions

In [ ]:
# Save training metrics (loss, accuracy, F1, etc.) as a JSON file
def save_metrics(metrics, output_dir):
    # Persisting metrics ensures the full training history is reproducible
    # and allows later comparison across runs or hyperparameter settings.
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)


# Plot and save loss and accuracy curves for the full training run
def save_loss_curves(metrics, output_dir):
    epochs = range(1, len(metrics["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curves reveal optimisation behaviour, helping diagnose underfitting,
    # overfitting, or instability across epochs.
    ax1.plot(epochs, metrics["train_loss"], label="Train")
    ax1.plot(epochs, metrics["val_loss"], label="Val")
    ax1.set_title("Loss")
    ax1.legend()

    # Accuracy curves provide an intuitive view of performance progression
    # and complement the loss curves for a fuller diagnostic picture.
    ax2.plot(epochs, metrics["train_acc"], label="Train")
    ax2.plot(epochs, metrics["val_acc"], label="Val")
    ax2.set_title("Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "loss_curves.png"))
    plt.close()


# Generate and save a confusion matrix heatmap
'''
def save_confusion_matrix(labels, preds, class_names, output_dir):
    # Confusion matrices expose systematic class‑pair confusions,
    # making them essential for targeted error analysis.
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(20, 18))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
    plt.close()
'''

# Main Training Logic


In [ ]:
# Example usage inside notebook:
# config = load_config("configs/experiments/convnext_t/phase1_lr1e-4_bs32.yaml")

def run_training(config):
    run_name   = config["run"]["name"]
    model_name = config["run"]["model"]

    # Each run gets its own directory so results, checkpoints, and configs remain isolated and reproducible.
    output_dir = config["output"]["base_dir"].replace("{model}", model_name)
    output_dir = os.path.join(output_dir, run_name)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Training {model_name} — {run_name}")

    # Reproducibility ensures identical behaviour across repeated runs.
    set_seed(config["dataset"]["seed"])

    # Device selection adapts automatically to available hardware.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    # Saving the resolved config guarantees that the exact training recipe is preserved.
    save_config(config, output_dir)

    # Data and model construction follow the modular pipeline defined earlier.
    train_loader, val_loader, test_loader = get_dataloaders(config)
    model = get_model(config).to(device)

    # Loss function setup
    label_smoothing = config["training"].get("label_smoothing", 0.0)
    use_sampler = config["training"].get("use_weighted_sampler", False)

    if use_sampler:
        # When using a sampler, class imbalance is handled at the sampling stage,
        # so the loss can remain unweighted.
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    else:
        # When not using a sampler, class weights counteract imbalance by penalising
        # mistakes on minority classes more heavily.
        targets = np.array(train_loader.dataset.targets)
        class_counts = np.bincount(
            targets,
            minlength=config["dataset"]["n_classes"]
        )
        class_counts = class_counts + 1e-6  # avoid division by zero
        weights = 1.0 / class_counts
        weights = weights / weights.mean()  # normalise for stability
        class_weights = torch.tensor(weights, dtype=torch.float32)

        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device),
            label_smoothing=label_smoothing
        )

    # Optimiser and scheduler define the optimisation dynamics.
    optimizer = get_optimizer(model, config)
    scheduler = get_scheduler(optimizer, config)

    # Mixed precision improves throughput on modern GPUs while reducing memory use.
    use_amp = config["training"].get("mixed_precision", False)
    scaler = torch.amp.GradScaler(device="cuda", enabled=use_amp)

    epochs = config["training"]["epochs"]
    log_interval = config["output"]["log_interval"]

    # Track best validation performance for checkpointing.
    best_val_f1 = 0.0
    metrics = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [], "val_f1": []
    }

    # Early stopping prevents overfitting and saves compute when progress stalls.
    patience = config["training"].get("early_stopping_patience", None)
    epochs_without_improvement = 0

    # Training loop
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        # Train for one epoch
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, log_interval,
            scaler=scaler, use_amp=use_amp
        )

        # Validate on the full validation set
        val_loss, val_acc, val_f1, val_preds, val_labels = validate(
            model, val_loader, criterion, device
        )

        # Scheduler step updates the LR according to the chosen schedule.
        scheduler.step()

        # Store metrics for plotting and analysis.
        metrics["train_loss"].append(train_loss)
        metrics["train_acc"].append(train_acc)
        metrics["val_loss"].append(val_loss)
        metrics["val_acc"].append(val_acc)
        metrics["val_f1"].append(val_f1)

        # Checkpointing ensures the best model (by F1) is saved, not just the final epoch.
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pth"))
            print(f"[Checkpoint] New best F1={best_val_f1:.4f}")

        else:
            epochs_without_improvement += 1
            print(f"[Early Stopping] No improvement for {epochs_without_improvement} epoch(s).")

            # Early stopping halts training when further epochs are unlikely to help.
            if patience is not None and epochs_without_improvement >= patience:
                print(f"[Early Stopping] Patience reached ({patience}). Stopping training early.")
                break

    # Save results for reproducibility and post‑hoc analysis.
    save_loss_curves(metrics, output_dir)
    save_confusion_matrix(val_labels, val_preds, train_loader.dataset.classes, output_dir)
    save_metrics(metrics, output_dir)

    print("Training complete.")

# Test run

In [40]:
# Load config
config = load_config("configs/experiments/smoke_test.yaml")

# Run training
run_training(config)

Training resnet50 — smoke_test
[Seed] Set to 42
[Device] cuda
[Imbalance Fix] Sampler DISABLED (Phase 1)
[Data] Train=7069 | Val=1754 | Test=1551
[Model] Loading resnet50 (pretrained=False)
[Model] Params — total: 23,565,404 | trainable: 23,565,404

Epoch 1/2


  Training:   0%|          | 0/884 [00:00<?, ?it/s]

[Checkpoint] New best F1=0.1199

Epoch 2/2


  Training:   0%|          | 0/884 [00:00<?, ?it/s]

[Checkpoint] New best F1=0.1743
Training complete.
